# 11.35 AlphaGo / AlphaZero / MuZero

Alpha-style systems combine learning with lookahead so policies and search improve together.

Reinforcement learning is where a prediction changes what data arrives next. Probability supplies transition probabilities and expectations; optimization supplies iterative improvement. These notebooks keep the environments tiny and CPU-only while implementing the real decision rule. Save a copy to Drive to edit.

In [ ]:

import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 1135
random.seed(SEED)
np.random.seed(SEED)

ACTIONS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
ACTION_NAMES = ["up", "right", "down", "left"]


@dataclass
class GridEnv:
    name: str
    height: int
    width: int
    start: tuple
    goal: tuple
    hazards: set
    traps: set
    slip: float
    wind: float
    budget: float
    max_steps: int
    reward_goal: float = 5.0
    reward_step: float = -0.05
    cost_hazard: float = 1.0

    @property
    def n_states(self):
        return self.height * self.width

    @property
    def n_actions(self):
        return 4

    def state_to_pos(self, state):
        row = state // self.width
        col = state % self.width
        return (row, col)

    def pos_to_state(self, pos):
        return pos[0] * self.width + pos[1]

    def in_bounds(self, pos):
        row, col = pos
        return 0 <= row < self.height and 0 <= col < self.width

    def start_state(self):
        return self.pos_to_state(self.start)

    def goal_state(self):
        return self.pos_to_state(self.goal)

    def transition(self, state, action, slip_override=None):
        slip = self.slip if slip_override is None else slip_override
        probs = []
        primary = self._move(state, action)
        probs.append((1.0 - slip, primary))
        side_actions = [(action + 1) % 4, (action - 1) % 4]
        for side in side_actions:
            probs.append((slip / 2.0, self._move(state, side)))
        merged = {}
        for prob, next_state in probs:
            merged[next_state] = merged.get(next_state, 0.0) + prob
        return list(merged.items())

    def _move(self, state, action):
        if state == self.goal_state():
            return state
        pos = np.array(self.state_to_pos(state))
        next_pos = tuple(pos + ACTIONS[action])
        if self.wind > 0 and action == 1:
            next_pos = (max(0, next_pos[0] - 1), next_pos[1])
        if not self.in_bounds(next_pos):
            next_pos = tuple(pos)
        return self.pos_to_state(next_pos)

    def reward_cost_done(self, state, action, next_state):
        pos = self.state_to_pos(next_state)
        done = next_state == self.goal_state()
        reward = self.reward_goal if done else self.reward_step
        cost = self.cost_hazard if pos in self.hazards or pos in self.traps else 0.0
        if pos in self.traps:
            reward -= 1.0
        return reward, cost, done


def make_f12_ladder():
    return [
        GridEnv("D1 two-state chain", 1, 2, (0, 0), (0, 1), {(0, 1)}, set(), 0.00, 0.0, 1.0, 3),
        GridEnv("D2 slippery 3-state hazards", 1, 3, (0, 0), (0, 2), {(0, 1)}, set(), 0.10, 0.0, 1.0, 6),
        GridEnv("D3 4x4 gridworld hazards", 4, 4, (3, 0), (0, 3), {(2, 1), (1, 2)}, set(), 0.08, 0.0, 1.2, 18),
        GridEnv("D4 stochastic windy grid", 5, 5, (4, 0), (0, 4), {(3, 1), (2, 2), (1, 3)}, set(), 0.15, 0.25, 1.4, 28),
        GridEnv("D5 sparse grid with traps", 6, 6, (5, 0), (0, 5), {(4, 1), (3, 2), (2, 3)}, {(1, 4), (4, 4)}, 0.18, 0.30, 1.5, 40),
    ]


def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)


def discounted_return(rewards, gamma=0.9):
    total = 0.0
    for step, reward in enumerate(rewards):
        total += (gamma ** step) * reward
    return total


def value_iteration(env, gamma=0.9, penalty=0.0, slip_override=None, iterations=80):
    values = np.zeros(env.n_states)
    q_values = np.zeros((env.n_states, env.n_actions))
    for _ in range(iterations):
        new_values = values.copy()
        for state in range(env.n_states):
            if state == env.goal_state():
                continue
            action_scores = []
            for action in range(env.n_actions):
                score = 0.0
                for prob, next_state in env.transition(state, action, slip_override):
                    reward, cost, done = env.reward_cost_done(state, action, next_state)
                    bootstrap = 0.0 if done else gamma * values[next_state]
                    score += prob * (reward - penalty * cost + bootstrap)
                action_scores.append(score)
            new_values[state] = max(action_scores)
            q_values[state] = action_scores
        values = new_values
    policy = np.argmax(q_values, axis=1)
    return values, q_values, policy


def evaluate_policy(env, policy, gamma=0.9, episodes=40, seed=0, slip_override=None):
    rng = np.random.default_rng(seed)
    returns = []
    costs = []
    wins = []
    paths = []
    for episode in range(episodes):
        state = env.start_state()
        rewards = []
        cost_values = []
        path = [state]
        for step in range(env.max_steps):
            action = int(policy[state])
            transitions = env.transition(state, action, slip_override)
            probs = np.array([item[0] for item in transitions])
            idx = rng.choice(len(transitions), p=probs)
            next_state = transitions[idx][1]
            reward, cost, done = env.reward_cost_done(state, action, next_state)
            rewards.append(reward)
            cost_values.append(cost)
            state = next_state
            path.append(state)
            if done:
                break
        returns.append(discounted_return(rewards, gamma))
        costs.append(sum(cost_values))
        wins.append(float(state == env.goal_state()))
        paths.append(path)
    return {
        "return": float(np.mean(returns)),
        "cost": float(np.mean(costs)),
        "win_rate": float(np.mean(wins)),
        "path": paths[0],
    }



def preview_ladder(ladder):
    rows = []
    for idx, env in enumerate(ladder, start=1):
        rows.append({
            "rung": f"D{idx}",
            "name": env.name,
            "shape": (env.height, env.width),
            "states": env.n_states,
            "slip": env.slip,
            "budget": env.budget,
            "hazards": len(env.hazards) + len(env.traps),
        })
    return rows


def plot_env_panel(ax, env, values, title):
    image = np.asarray(values).reshape(env.height, env.width)
    ax.imshow(image, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    for hazard in env.hazards:
        ax.text(hazard[1], hazard[0], "H", color="white", ha="center", va="center")
    for trap in env.traps:
        ax.text(trap[1], trap[0], "T", color="red", ha="center", va="center")
    ax.text(env.start[1], env.start[0], "S", color="white", ha="center", va="center")
    ax.text(env.goal[1], env.goal[0], "G", color="white", ha="center", va="center")


## The concept, built once (D1)

Alpha-style planning uses $$V^\pi(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)(R(s,a,s')+\gamma V^\pi(s'))$$ with a policy prior, value estimate, and dynamics model. Exact lesson arithmetic is asserted before search.

In [ ]:

rewards = [1, 0, 2]
gamma = 0.9
G = discounted_return(rewards, gamma)
y = 1 + gamma * 0.8
q_new = 0.4 + 0.5 * (y - 0.4)
probs = softmax([1, 0])
expected_reward = probs[0] * 2 + probs[1] * 0
ucb = 0.55 + math.sqrt(2 * math.log(20) / 5)
print("G", round(G, 3))
print("target", round(y, 3))
print("Q_new", round(q_new, 3))
print("policy", np.round(probs, 3))
print("expected reward", round(float(expected_reward), 3))
print("UCB", round(ucb, 3))
assert round(G, 3) == 2.620
assert round(y, 3) == 1.720
assert round(q_new, 3) == 1.060
assert round(float(probs[0]), 3) == 0.731
assert round(float(probs[1]), 3) == 0.269
assert round(float(expected_reward), 3) == 1.462
assert round(ucb, 3) == 1.645


On D1, PUCT-style MCTS backs up value-guided returns through a tiny game tree, the AlphaZero pattern in miniature.

In [ ]:
class TinyGame:
    def __init__(self, env):
        self.env = env
        self.start = env.start_state()

    def legal_actions(self, state):
        return list(range(self.env.n_actions))

    def next_state_reward_done(self, state, action):
        transitions = self.env.transition(state, action, slip_override=0.0)
        next_state = max(transitions, key=lambda item: item[0])[1]
        reward, cost, done = self.env.reward_cost_done(state, action, next_state)
        return next_state, reward, done


def heuristic_value(env, state, gamma=0.9):
    pos = env.state_to_pos(state)
    dist = abs(pos[0] - env.goal[0]) + abs(pos[1] - env.goal[1])
    return (gamma ** dist) * env.reward_goal


def prior_policy(env, state):
    scores = []
    pos = np.array(env.state_to_pos(state))
    goal = np.array(env.goal)
    for action in range(env.n_actions):
        next_pos = pos + ACTIONS[action]
        if not env.in_bounds(tuple(next_pos)):
            next_pos = pos
        distance = np.sum(np.abs(next_pos - goal))
        scores.append(-float(distance))
    return softmax(scores)


def mcts_with_value_policy(env, simulations=35, gamma=0.9, c_puct=1.4, depth_limit=8):
    game = TinyGame(env)
    root = env.start_state()
    visits = {}
    value_sum = {}
    priors = {}
    children = {}

    def ensure(state):
        if state not in priors:
            priors[state] = prior_policy(env, state)
            children[state] = {}
            for action in game.legal_actions(state):
                next_state, reward, done = game.next_state_reward_done(state, action)
                children[state][action] = (next_state, reward, done)
                visits[(state, action)] = 0
                value_sum[(state, action)] = 0.0

    for simulation in range(simulations):
        state = root
        path = []
        for depth in range(depth_limit):
            ensure(state)
            total_visits = 1 + sum(visits[(state, action)] for action in game.legal_actions(state))
            scores = []
            for action in game.legal_actions(state):
                n = visits[(state, action)]
                q = 0.0 if n == 0 else value_sum[(state, action)] / n
                u = c_puct * priors[state][action] * math.sqrt(total_visits) / (1 + n)
                scores.append(q + u)
            action = int(np.argmax(scores))
            next_state, reward, done = children[state][action]
            path.append((state, action, reward))
            state = next_state
            if done:
                leaf_value = reward
                break
        else:
            leaf_value = heuristic_value(env, state, gamma=gamma)
        backup = leaf_value
        for state, action, reward in reversed(path):
            visits[(state, action)] += 1
            backup = reward + gamma * backup
            value_sum[(state, action)] += backup
    ensure(root)
    root_counts = np.array([visits[(root, action)] for action in game.legal_actions(root)], dtype=float)
    if np.sum(root_counts) == 0:
        root_policy = np.ones(env.n_actions) / env.n_actions
    else:
        root_policy = root_counts / np.sum(root_counts)
    policy = np.zeros(env.n_states, dtype=int)
    values = np.zeros(env.n_states)
    for state in range(env.n_states):
        ensure(state)
        action_scores = []
        for action in game.legal_actions(state):
            n = visits[(state, action)]
            q = heuristic_value(env, children[state][action][0], gamma=gamma) if n == 0 else value_sum[(state, action)] / n
            action_scores.append(q)
        policy[state] = int(np.argmax(action_scores))
        values[state] = max(action_scores)
    return policy, values, root_policy, visits


def immediate_reward_search(env):
    policy = np.zeros(env.n_states, dtype=int)
    values = np.zeros(env.n_states)
    for state in range(env.n_states):
        scores = []
        for action in range(env.n_actions):
            score = 0.0
            for prob, next_state in env.transition(state, action, slip_override=0.0):
                reward, cost, done = env.reward_cost_done(state, action, next_state)
                score += prob * reward
            scores.append(score)
        policy[state] = int(np.argmax(scores))
        values[state] = max(scores)
    return policy, values

env = make_f12_ladder()[0]
policy, values, root_policy, visits = mcts_with_value_policy(env, simulations=20)
metrics = evaluate_policy(env, policy, seed=SEED)
print("root policy", np.round(root_policy, 3))
print("metrics", metrics)
assert len(root_policy) == env.n_actions

## The dataset ladder

Family F12 uses a D1–D5 sequential-decision ladder inline: a two-state chain, slippery chain, 4x4 grid, windy grid, and sparse trap grid.

In [ ]:
ladder = make_f12_ladder()
for row in preview_ladder(ladder):
    print(row)
print("sample D5 policy grid shape", (ladder[-1].height, ladder[-1].width))

## Run the SAME method across D1-D5

Apply the same method to every rung and collect the plan metric.

In [ ]:
results = []
artifacts = []
for index, env in enumerate(ladder, start=1):
    policy, values, root_policy, visits = mcts_with_value_policy(env, simulations=25 + 5 * index)
    metrics = evaluate_policy(env, policy, seed=SEED + index, slip_override=0.0)
    shallow_policy, shallow_values = immediate_reward_search(env)
    shallow_metrics = evaluate_policy(env, shallow_policy, seed=SEED + index, slip_override=0.0)
    row = {
        "rung": f"D{index}",
        "return": metrics["return"],
        "win_rate": metrics["win_rate"],
        "shallow_return": shallow_metrics["return"],
        "root_best_prob": float(np.max(root_policy)),
    }
    results.append(row)
    artifacts.append((env, values, shallow_values, root_policy))
print("rung | MCTS return | shallow return | win_rate | root best prob")
for row in results:
    print(row["rung"], round(row["return"], 3), round(row["shallow_return"], 3), round(row["win_rate"], 3), round(row["root_best_prob"], 3))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, (env, values, shallow_values, root_policy) in enumerate(artifacts):
    plot_env_panel(axes[0, col], env, values, f"D{col + 1} MCTS value")
    axes[1, col].bar(ACTION_NAMES, root_policy)
    axes[1, col].set_ylim(0, 1)
    axes[1, col].set_title(f"D{col + 1} root policy")
    axes[1, col].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
xs = np.arange(1, 6)
plt.plot(xs, [row["return"] for row in results], marker="o", label="MCTS + value")
plt.plot(xs, [row["shallow_return"] for row in results], marker="s", label="immediate reward")
plt.xticks(xs, [row["rung"] for row in results])
plt.xlabel("rung")
plt.ylabel("return")
plt.title("Search-guided return across D1-D5")
plt.legend()
plt.show()

## Pitfall on D5: confusing reward with return

A shallow search that sees only immediate reward misses delayed goal value. Value-backed discounted lookahead fixes the objective by backing up future consequences.

In [ ]:
env = ladder[-1]
wrong_policy, wrong_values = immediate_reward_search(env)
wrong = evaluate_policy(env, wrong_policy, seed=88, slip_override=0.0)
fixed_policy, fixed_values, root_policy, visits = mcts_with_value_policy(env, simulations=55)
fixed = evaluate_policy(env, fixed_policy, seed=88, slip_override=0.0)
print("wrong shallow", wrong)
print("fixed MCTS", fixed)
print("return lift", round(fixed["return"] - wrong["return"], 3))

## Evaluate it + Practice

- Compare the reported return / win-rate against a no-skill baseline such as a random or immediate-reward policy.
- Sanity check that the D1 result matches the exact lesson arithmetic before trusting harder rungs.
- Ablate the key idea, such as removing constraints, randomization, return conditioning, population replay, or search.
- Watch failure signals: budget violations, transfer collapse, unsupported target returns, non-stationary opponents, or shallow reward chasing.

Practice:
1. Change the discount from 0.9 to 0.8 and predict which rung changes most.


2. Add one hazard or trap to D4 and rerun only the small table, not a long training job.

3. Replace the no-skill baseline with a hand-written safe policy and compare the metric.